# Black-and-white plot versions for print

Reads the same cached predictions as the color notebooks and re-renders everything into `../plots/bw/` with:
- SPROUT spider → 2×3 grid of per-benchmark bar charts comparing CARROT at 10 %/20 %/30 % of GPT-4o cost vs GPT-4o baseline.
- Routing curves (plots 2, 3, 4, 5) → black lines distinguished by linestyle + markers.
- PCA / kPCA / Isomap dim plots → plain black.

All lines use `color='black'`; distinctions are linestyle and marker only.

In [ ]:
import os, sys
sys.path.insert(0, '../carrot/')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import pi
from matplotlib.ticker import MaxNLocator
from constants import OPEN_MODELS, OPEN_COSTS
from utils import route, route_pairwise, route_routerbench

BW_DIR = '../plots/bw'
os.makedirs(BW_DIR, exist_ok=True)

# Distinct linestyles for up to 5 routers per plot
LINESTYLES = {
    'mf':              (0, (1, 1)),            # densely dotted
    'rorf':            (0, (5, 1)),            # densely dashed
    'roberta-binary':  (0, (3, 1, 1, 1)),      # densely dashdotted
    'carrot-knn':      '--',                   # dashed
    'carrot-roberta':  '-',                    # solid
    # plot 3 extras
    'carrot-knn-rb':   (0, (3, 3)),            # mid-dashed
    'carrot-roberta-rb': (0, (1, 1, 3, 1)),   # sparse dot-dash
    # plot 4
    'CARROT':          '-',
    'Routerbench':     '--',
    'zero_router':     ':',
}
MARKERS_R = {
    'mf': 'o', 'rorf': 's', 'roberta-binary': '^',
    'carrot-knn': 'D', 'carrot-roberta': 'v',
    'carrot-knn-rb': '>', 'carrot-roberta-rb': '<',
    'CARROT': 'o', 'Routerbench': 's', 'zero_router': '^',
}
LABELS_R = {
    'mf': 'RouteLLM (MF)',
    'rorf': 'Not-Diamond RoRF',
    'roberta-binary': 'RouteLLM (RoBERTa)',
    'carrot-knn': 'CARROT (KNN)',
    'carrot-roberta': 'CARROT (RoBERTa)',
    'carrot-knn-rb': 'Routerbench (KNN)',
    'carrot-roberta-rb': 'Routerbench (RoBERTa)',
}

# Scatter markers for individual model points (shared across plots)
SCATTER_MARKERS = ['o', 's', 'D', '^', 'v', 'p', '*', 'x', '+', 'h', 'H', 'd', '>', 'P', '<', '|', '_', '.', ',', '1', '2']

MAX_XTICKS = 4  # cap routing-curve x-axis labels for readability

def plot_curve(ax, c, p, key, every=50):
    """Draw a routing curve in black using linestyle-only distinction, with sparse markers."""
    ax.plot(c, p, color='black', linestyle=LINESTYLES[key], linewidth=1.2,
            marker=MARKERS_R.get(key, None), markersize=4, markevery=max(1, len(c)//10),
            label=LABELS_R.get(key, key))

## Plot 1 — SPROUT: CARROT vs GPT-4o (bar-grid replacement for spider)

In [ ]:
data = np.load('../data/sprout/IBMMIX_o3mini.npy', allow_pickle=True).item()
prompts = data['prompts']
categories = data['categories']
models = [k for k in data.keys() if k not in ('prompts', 'categories')]
gpt4o_idx = models.index('openai-gpt-4o')

cost_true = np.stack([np.asarray(data[m]['actual cost']) for m in models], axis=1)
cost_pred = np.stack([np.asarray(data[m]['predicted cost']) for m in models], axis=1)
perf_true = np.stack([np.asarray(data[m]['actual perf']) for m in models], axis=1)
perf_pred = np.stack([np.asarray(data[m]['predicted perf']) for m in models], axis=1)
perf_pred_sig = 1 / (1 + np.exp(-perf_pred))

def route_sprout(scores, cost_actual, cost_pred, correctness, lamb_range=np.arange(0, 1.001, 0.001)):
    router_cost = np.zeros((scores.shape[0], len(lamb_range)))
    router_perf = np.zeros_like(router_cost)
    for i, lam in enumerate(lamb_range):
        idx = ((1 - lam) * scores - lam * cost_pred * 1000).argmax(axis=1, keepdims=True)
        router_perf[:, i] = np.take_along_axis(correctness, idx, axis=1).reshape(-1)
        router_cost[:, i] = np.take_along_axis(cost_actual, idx, axis=1).reshape(-1)
    return router_cost, router_perf

router_cost, router_perf = route_sprout(perf_pred_sig, cost_true, cost_pred, perf_true)

data_names = ['gpqa', 'MuSR', 'MMLU-Pro', 'MATH', 'openhermes', 'ragbench']
props = [0.1, 0.2, 0.3]

# For each benchmark, compute CARROT's best attainable accuracy under each cost budget
# (expressed as a fraction of GPT-4o's cost on that benchmark), and GPT-4o accuracy.
bench_rows = []
for cat in data_names:
    mask = np.where(np.char.find(np.asarray(categories), cat) >= 0)[0]
    cost_mean = cost_true[mask, :].mean(axis=0)
    perf_mean = perf_true[mask, :].mean(axis=0)
    rc_mean = router_cost[mask, :].mean(axis=0)
    rp_mean = router_perf[mask, :].mean(axis=0)
    gpt_cost = cost_mean[gpt4o_idx]
    gpt_perf = perf_mean[gpt4o_idx]
    row = {'benchmark': cat, 'gpt4o_acc': gpt_perf}
    for p in props:
        # Best CARROT accuracy at cost <= p * gpt4o_cost
        under = rc_mean <= p * gpt_cost
        row[f'carrot_{int(p*100)}'] = float(rp_mean[under].max()) if under.any() else np.nan
    bench_rows.append(row)
bench_df = pd.DataFrame(bench_rows).set_index('benchmark')
print(bench_df.round(3))

# 2x3 grid of bar charts
fig, axes = plt.subplots(2, 3, figsize=(9, 5.5), sharey=True)
bar_labels = ['CARROT\n10%', 'CARROT\n20%', 'CARROT\n30%', 'GPT-4o\n(100%)']
hatches = ['//', '\\\\', 'xx', '']  # hatch patterns for 4 bars
for ax, cat in zip(axes.flat, data_names):
    row = bench_df.loc[cat]
    vals = [row['carrot_10'], row['carrot_20'], row['carrot_30'], row['gpt4o_acc']]
    positions = np.arange(4)
    bars = ax.bar(positions, vals, color='white', edgecolor='black', linewidth=1)
    for bar, h in zip(bars, hatches):
        bar.set_hatch(h)
    # draw horizontal reference line at GPT-4o accuracy
    ax.axhline(row['gpt4o_acc'], color='black', linestyle=':', linewidth=0.8)
    ax.set_xticks(positions)
    ax.set_xticklabels(bar_labels, fontsize=8)
    ax.set_title(cat, fontsize=10)
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_ylim(0, 1.02)
    for pos, v in zip(positions, vals):
        ax.text(pos, v + 0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=7)

for ax in axes[:, 0]:
    ax.set_ylabel('Accuracy')
fig.suptitle('SPROUT — CARROT at budget fractions of GPT-4o cost vs GPT-4o', fontsize=11)
fig.tight_layout()
fig.savefig(f'{BW_DIR}/sprout_spider_bars.pdf', bbox_inches='tight')
plt.show()

## Plot 2 — Routerbench: CARROT vs binary routers

In [ ]:
PREDS = '../data/routerbench/preds'
meta = np.load(f'{PREDS}/meta.npy', allow_pickle=True).item()
models = meta['models']
Y_test = meta['Y_test']
C_test = meta['C_test']
small_ind = meta['small_model_ind']
large_ind = meta['large_model_ind']

def load(p, name):
    path = f'{p}/{name}.npy'
    return np.load(path, allow_pickle=True) if os.path.exists(path) else None

Y_hat = {m: load(PREDS, f'Y_hat_{m}') for m in ['mf', 'rorf', 'roberta-binary', 'carrot-knn', 'carrot-roberta']}
C_hat = {m: load(PREDS, f'C_hat_{m}') for m in ['carrot-knn', 'carrot-roberta']}

mult = 100
curves = {}
for name in ['mf', 'rorf', 'roberta-binary']:
    if Y_hat[name] is None: continue
    curves[name] = route_pairwise(np.asarray(Y_hat[name]).squeeze(), C_test, Y_test, large_ind, small_ind)
for name in ['carrot-knn', 'carrot-roberta']:
    if Y_hat[name] is None or C_hat[name] is None: continue
    curves[name] = route(Y_hat[name], C_test, mult * C_hat[name], Y_test)

LABEL_MODELS_RB = {'gpt-4-1106-preview', 'zero-one-ai/Yi-34B-Chat',
                   'gpt-3.5-turbo-1106', 'mistralai/mixtral-8x7b-chat'}

fig, ax = plt.subplots(1, 1, figsize=(4.6, 4.3))
for name, (c, p) in curves.items():
    plot_curve(ax, c, p, name)
for i, m in enumerate(models):
    x, y = C_test[:, i].mean(0), Y_test[:, i].mean(0)
    ax.scatter([x], [y], marker=SCATTER_MARKERS[i % len(SCATTER_MARKERS)],
               edgecolor='black', facecolor='white', s=22, linewidths=0.8)
    if m in LABEL_MODELS_RB:
        ax.annotate(m.split('/')[-1], (x, y), size=6)
ax.set_title('Routerbench — CARROT vs binary routers')
ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.xaxis.set_major_locator(MaxNLocator(nbins=MAX_XTICKS - 1))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.savefig(f'{BW_DIR}/routerbench_binary.pdf', bbox_inches='tight')
plt.show()

## Plot 3 — Routerbench: CARROT vs Routerbench routing rule

In [ ]:
curves3 = {}
for base in ['carrot-knn', 'carrot-roberta']:
    if Y_hat[base] is None or C_hat[base] is None: continue
    curves3[base] = route(Y_hat[base], C_test, mult * C_hat[base], Y_test)
    rb_key = {'carrot-knn': 'carrot-knn-rb', 'carrot-roberta': 'carrot-roberta-rb'}[base]
    curves3[rb_key] = route_routerbench(Y_hat[base], C_test, Y_test)

fig, ax = plt.subplots(1, 1, figsize=(4.6, 4.3))
for name, (c, p) in curves3.items():
    plot_curve(ax, c, p, name)
for i, m in enumerate(models):
    x, y = C_test[:, i].mean(0), Y_test[:, i].mean(0)
    ax.scatter([x], [y], marker=SCATTER_MARKERS[i % len(SCATTER_MARKERS)],
               edgecolor='black', facecolor='white', s=22, linewidths=0.8)
    if m in LABEL_MODELS_RB:
        ax.annotate(m.split('/')[-1], (x, y), size=6)
ax.set_title('Routerbench — CARROT vs Routerbench router')
ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.xaxis.set_major_locator(MaxNLocator(nbins=MAX_XTICKS - 1))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.savefig(f'{BW_DIR}/routerbench_vs_rb.pdf', bbox_inches='tight')
plt.show()

## Plot 4 — SPROUT: CARROT vs Routerbench rule vs zero router

In [ ]:
data4 = np.load('../data/sprout/IBMMIX_o3mini.npy', allow_pickle=True).item()
all_models = [k for k in data4.keys() if k not in ('prompts', 'categories')]
scores = np.stack([np.asarray(data4[m]['actual perf']) for m in all_models], axis=1)
costs = np.stack([np.asarray(data4[m]['actual cost']) for m in all_models], axis=1)
p_logits = np.stack([np.asarray(data4[m]['predicted perf']) for m in all_models], axis=1)
p_costs = np.stack([np.asarray(data4[m]['predicted cost']) for m in all_models], axis=1)
p_scores = 1 / (1 + np.exp(-p_logits))
avg_costs = costs.mean(axis=0)

lamb = np.arange(0, 1.001, 0.01)
rc = np.zeros((scores.shape[0], len(lamb)))
rp = np.zeros_like(rc); bc = np.zeros_like(rc); bp = np.zeros_like(rc)
for i, lm in enumerate(lamb):
    ci = ((1 - lm) * p_scores - lm * p_costs * 100).argmax(axis=1, keepdims=True)
    rbi = ((1 - lm) * p_scores - lm * avg_costs[None, :] * 100).argmax(axis=1, keepdims=True)
    rp[:, i] = np.take_along_axis(scores, ci, axis=1).reshape(-1)
    rc[:, i] = np.take_along_axis(costs, ci, axis=1).reshape(-1)
    bp[:, i] = np.take_along_axis(scores, rbi, axis=1).reshape(-1)
    bc[:, i] = np.take_along_axis(costs, rbi, axis=1).reshape(-1)

cost_mean = costs.mean(axis=0)
score_mean = scores.mean(axis=0)
zero_names = ['wxai-granite-3-8b-instruct-8k-max-tokens', 'openai-gpt-4o-mini', 'openai-o3-mini']
zi = sorted([all_models.index(m) for m in zero_names if m in all_models], key=lambda k: cost_mean[k])

SKIP_LABEL_MODELS = {'wxai-llama-3-3-70b-instruct', 'wxai-llama-3-1-70b-instruct',
                     'wxai-llama-3-1-8b-instruct'}

fig, ax = plt.subplots(1, 1, figsize=(4.6, 4.3))
ax.plot(rc.mean(0), rp.mean(0), color='black', linestyle=LINESTYLES['CARROT'], linewidth=1.3, marker='o', markevery=10, markersize=4, label='CARROT')
ax.plot(bc.mean(0), bp.mean(0), color='black', linestyle=LINESTYLES['Routerbench'], linewidth=1.3, marker='s', markevery=10, markersize=4, label='Routerbench')
ax.plot(cost_mean[zi], score_mean[zi], color='black', linestyle=LINESTYLES['zero_router'], linewidth=1.3, marker='^', markersize=5, label='zero router')
for i, m in enumerate(all_models):
    x, y = cost_mean[i], score_mean[i]
    ax.scatter([x], [y], marker=SCATTER_MARKERS[i % len(SCATTER_MARKERS)],
               edgecolor='black', facecolor='white', s=22, linewidths=0.8)
    if m not in SKIP_LABEL_MODELS:
        ax.annotate(m, (x, y), size=5)
ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.60, 0.95)
ax.xaxis.set_major_locator(MaxNLocator(nbins=MAX_XTICKS - 1))
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
fig.savefig(f'{BW_DIR}/sprout_vs_rb_zero.pdf', bbox_inches='tight')
plt.show()

## Plot 5 — Open-LLM-Leaderboard v2: CARROT vs binary routers

In [ ]:
PREDS5 = '../data/open-llm-lb-v2/preds'
meta5 = np.load(f'{PREDS5}/meta.npy', allow_pickle=True).item()
models5 = meta5['models']
Y_test5 = meta5['Y_test']
IT_test5 = meta5['IT_test']
small_ind5 = meta5['small_model_ind']
large_ind5 = meta5['large_model_ind']
mm = [m.replace('open-llm-leaderboard/', '').replace('__', '/').replace('-details', '') for m in OPEN_MODELS]
model_order = [np.argmax(np.array(models5) == m) for m in mm if np.sum(np.array(models5) == m) > 0]
cost_rates = np.array([OPEN_COSTS[m] / 1e6 for m in models5])[None, :]
C_test5 = IT_test5[:, model_order] * cost_rates

Y_hat5 = {m: load(PREDS5, f'Y_hat_{m}') for m in ['mf', 'rorf', 'roberta-binary', 'carrot-knn', 'carrot-roberta']}

curves5 = {}
for name in ['mf', 'rorf', 'roberta-binary']:
    if Y_hat5[name] is None: continue
    curves5[name] = route_pairwise(np.asarray(Y_hat5[name]).squeeze(), C_test5, Y_test5, large_ind5, small_ind5)
for name in ['carrot-knn', 'carrot-roberta']:
    if Y_hat5[name] is None: continue
    curves5[name] = route(Y_hat5[name], C_test5, C_test5, Y_test5)

LABEL_MODELS_OPEN = {'Qwen/Qwen2.5-72B-Instruct', 'Qwen/Qwen2-72B-Instruct',
                     'alpindale/WizardLM-2-8x22B', 'mistralai/Mistral-7B-Instruct-v0.3',
                     'google/gemma-2b-it'}

fig, ax = plt.subplots(1, 1, figsize=(4.6, 4.3))
for name, (c, p) in curves5.items():
    plot_curve(ax, c, p, name)
for i, m in enumerate(models5):
    x, y = C_test5[:, i].mean(0), Y_test5[:, i].mean(0)
    ax.scatter([x], [y], marker=SCATTER_MARKERS[i % len(SCATTER_MARKERS)],
               edgecolor='black', facecolor='white', s=22, linewidths=0.8)
    if m in LABEL_MODELS_OPEN:
        ax.annotate(m.split('/')[-1], (x, y), size=6)
ax.set_title('Open-LLM-Leaderboard v2 — CARROT vs binary routers')
ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.set_xlim(left=0)
ax.xaxis.set_major_locator(MaxNLocator(nbins=MAX_XTICKS - 1))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.savefig(f'{BW_DIR}/openllm_binary.pdf', bbox_inches='tight')
plt.show()

## Dim-analysis plots (PCA / kPCA / Isomap) in BW

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import Isomap

d = np.load('../data/sprout/sprout_data_train_test.npy', allow_pickle=True).item()
X = np.vstack([np.asarray(d['XOAI_train']).astype(np.float64), np.asarray(d['XOAI_test']).astype(np.float64)])
Xc = StandardScaler(with_mean=True, with_std=False).fit_transform(X)
RNG = np.random.default_rng(42)

# PCA
pca = PCA().fit(Xc)
eigvals = pca.explained_variance_
cum = np.cumsum(pca.explained_variance_ratio_)
TOP = 100
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(range(1, TOP + 1), eigvals[:TOP], color='black', marker='o', ms=2.5, lw=1)
axes[0].set_xlabel('component index'); axes[0].set_ylabel('eigenvalue')
axes[0].set_title(f'Scree plot (top {TOP})'); axes[0].grid(True, alpha=0.3)
axes[1].plot(range(1, len(cum) + 1), cum, color='black', lw=1)
for t in [0.9, 0.95, 0.99]:
    dn = int(np.searchsorted(cum, t) + 1)
    axes[1].axhline(t, color='black', linestyle=':', lw=0.6)
    axes[1].axvline(dn, color='black', linestyle=':', lw=0.6)
    axes[1].annotate(f'{int(t*100)}% @ {dn}', xy=(dn, t), xytext=(dn + 20, t - 0.04), fontsize=8)
axes[1].set_xlabel('component index'); axes[1].set_ylabel('cumulative variance')
axes[1].set_title('Cumulative variance'); axes[1].set_ylim(0, 1.02); axes[1].grid(True, alpha=0.3)
fig.suptitle(f'SPROUT OpenAI embedding — PCA  (n={X.shape[0]}, d=1536)')
fig.tight_layout(); fig.savefig(f'{BW_DIR}/sprout_pca_scree.pdf', bbox_inches='tight'); plt.show()

# Kernel PCA (RBF)
sub_idx = RNG.choice(Xc.shape[0], size=min(5000, Xc.shape[0]), replace=False)
kpca = KernelPCA(kernel='rbf', n_components=50, gamma=1.0/X.shape[1], random_state=42).fit(Xc[sub_idx])
ev = np.sort(np.asarray(getattr(kpca, 'eigenvalues_', getattr(kpca, 'lambdas_', None))))[::-1]
fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5))
ax.plot(range(1, len(ev) + 1), ev, color='black', marker='o', ms=3, lw=1)
ax.set_xlabel('component index'); ax.set_ylabel('eigenvalue')
ax.set_title(f'RBF Kernel PCA scree  (γ=1/d, n={len(sub_idx)})'); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(f'{BW_DIR}/sprout_kpca_scree.pdf', bbox_inches='tight'); plt.show()

# Isomap
sub_idx = RNG.choice(Xc.shape[0], size=min(3000, Xc.shape[0]), replace=False)
ks = [2, 5, 10, 20, 50]
recon = [Isomap(n_components=k, n_neighbors=15).fit(Xc[sub_idx]).reconstruction_error() for k in ks]
fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5))
ax.plot(ks, recon, color='black', marker='o', lw=1)
ax.set_xlabel('Isomap latent dim k'); ax.set_ylabel('reconstruction error')
ax.set_title(f'Isomap reconstruction error  (n={len(sub_idx)})'); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(f'{BW_DIR}/sprout_isomap_recon.pdf', bbox_inches='tight'); plt.show()